# StockSentinel — Full Pipeline on Google Colab
### Run all cells top to bottom. At the end, download one zip file.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save

**What this notebook does:**
1. Installs all dependencies
2. Downloads 6 years of price data (yfinance)
3. Engineers technical indicators
4. Downloads news headlines (automatic via requests)
5. Runs FinBERT sentiment on all 5 tickers
6. Builds sliding-window sequences
7. Trains LSTM+Sentiment, LSTM-Only, ARIMA, SVR
8. Evaluates all models → metrics.json
9. Packages everything into `stocksentinel_results.zip` for download

## Cell 1 — Check GPU & Install Dependencies

In [ ]:
import subprocess, sys

# Verify GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✓ GPU detected:')
    print(result.stdout.split('\n')[8] if len(result.stdout.split('\n')) > 8 else result.stdout[:200])
else:
    print('⚠ No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

# Install packages not pre-installed on Colab
!pip install yfinance pmdarima transformers accelerate -q
print('✓ Dependencies installed')

## Cell 2 — Create Folder Structure

In [ ]:
from pathlib import Path

dirs = [
    'raw_data/prices', 'raw_data/news',
    'processed_data', 'sentiment_scores',
    'saved_models',
    'results/training_history', 'results/predictions',
]
for d in dirs:
    Path(d).mkdir(parents=True, exist_ok=True)

print('✓ Folder structure ready')

## Cell 3 — Download Price Data (yfinance)

In [ ]:
import yfinance as yf
import pandas as pd

TICKERS = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'GOOGL']
START, END = '2018-01-01', '2024-12-31'

for ticker in TICKERS:
    df = yf.download(ticker, start=START, end=END, auto_adjust=True, progress=False)
    df.reset_index(inplace=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] for c in df.columns]
    df.to_csv(f'raw_data/prices/{ticker}_2018_2024.csv', index=False)
    print(f'  {ticker}: {len(df)} rows')

print('✓ Price data downloaded')

## Cell 4 — Download News Headlines
Uses the Kaggle API. If you don't have a Kaggle account, the cell falls back to
a smaller public dataset fetched directly — sentiment will still work, just fewer headlines.

In [ ]:
import os, json, requests
from pathlib import Path

# ── Option A: Kaggle API (best dataset, ~2M articles) ──────────────────────────
# Paste your Kaggle username and key here (from kaggle.com → Account → API → Create New Token)
KAGGLE_USERNAME = ''   # e.g. 'aniqamrin'
KAGGLE_KEY      = ''   # e.g. 'abc123def456...'

kaggle_ok = False

if KAGGLE_USERNAME and KAGGLE_KEY:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
        json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    !pip install kaggle -q
    ret = os.system('kaggle datasets download -d miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests -p raw_data/news --unzip')
    if ret == 0:
        kaggle_ok = True
        print('✓ Kaggle dataset downloaded')

# ── Option B: Fallback — download a curated subset via Hugging Face datasets ──
if not kaggle_ok:
    print('Kaggle credentials not set — using HuggingFace financial-news dataset as fallback...')
    !pip install datasets -q
    from datasets import load_dataset
    ds = load_dataset('zeroshot/twitter-financial-news-sentiment', split='train', trust_remote_code=True)
    df_hf = ds.to_pandas()
    print(f'  Loaded {len(df_hf)} financial news records')
    print(f'  Columns: {list(df_hf.columns)}')
    # Save as generic headlines file — filter_news will pick it up
    df_hf.to_csv('raw_data/news/kaggle_financial_news.csv', index=False)
    print('✓ Fallback news dataset saved')

## Cell 5 — Filter News by Ticker

In [ ]:
import pandas as pd
from pathlib import Path

TICKERS = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'GOOGL']

# Find the downloaded CSV
news_dir = Path('raw_data/news')
candidates = list(news_dir.glob('*.csv'))
print(f'News files found: {[f.name for f in candidates]}')

# Column name aliases across different dataset schemas
TICKER_ALIASES   = ['stock_symbol', 'ticker', 'symbol', 'Stock Symbol', 'label', 'tic']
DATE_ALIASES     = ['date', 'publish_date', 'publishedAt', 'Date', 'time']
HEADLINE_ALIASES = ['headline', 'title', 'text', 'Title', 'Headline', 'sentence']

def _col(df, aliases):
    for a in aliases:
        if a in df.columns: return a
    return None

for csv_path in candidates:
    print(f'\nProcessing {csv_path.name} ...')
    try:
        df = pd.read_csv(csv_path, low_memory=False)
    except Exception as e:
        print(f'  [skip] {e}'); continue

    ticker_col   = _col(df, TICKER_ALIASES)
    date_col     = _col(df, DATE_ALIASES)
    headline_col = _col(df, HEADLINE_ALIASES)

    if not all([ticker_col, date_col, headline_col]):
        print(f'  Columns: {list(df.columns)}')
        print('  [warn] Cannot auto-detect columns — skipping')
        continue

    df[date_col] = pd.to_datetime(df[date_col], errors='coerce', utc=True)
    df = df.dropna(subset=[date_col, headline_col])
    df[date_col] = df[date_col].dt.tz_localize(None)

    for ticker in TICKERS:
        subset = df[df[ticker_col].astype(str).str.upper() == ticker]
        subset = subset[[date_col, headline_col]].rename(columns={date_col:'date', headline_col:'headline'})
        subset = subset.sort_values('date').drop_duplicates()
        subset = subset[(subset['date'] >= '2018-01-01') & (subset['date'] <= '2024-12-31')]
        out = news_dir / f'{ticker}_headlines.csv'
        if len(subset) > 0:
            if out.exists():
                existing = pd.read_csv(out, parse_dates=['date'])
                subset = pd.concat([existing, subset]).drop_duplicates().sort_values('date')
            subset.to_csv(out, index=False)
            print(f'  {ticker}: {len(subset):,} headlines')

# Check coverage
print('\nFinal headline counts:')
for t in TICKERS:
    p = news_dir / f'{t}_headlines.csv'
    n = len(pd.read_csv(p)) if p.exists() else 0
    print(f'  {t}: {n:,}')

## Cell 6 — Engineer Technical Features

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

TICKERS = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'GOOGL']

def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=period-1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period-1, min_periods=period).mean()
    return 100 - (100 / (1 + avg_gain / (avg_loss + 1e-10)))

def compute_macd(series, fast=12, slow=26, signal=9):
    macd = series.ewm(span=fast, adjust=False).mean() - series.ewm(span=slow, adjust=False).mean()
    return macd, macd.ewm(span=signal, adjust=False).mean()

def compute_bb(series, period=20):
    sma = series.rolling(period).mean()
    std = series.rolling(period).std()
    return sma + 2*std, sma - 2*std

for ticker in TICKERS:
    df = pd.read_csv(f'raw_data/prices/{ticker}_2018_2024.csv', parse_dates=['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    df['Close']  = pd.to_numeric(df['Close'],  errors='coerce').ffill()
    df['Volume'] = pd.to_numeric(df['Volume'], errors='coerce').fillna(0)

    df['log_return']  = np.log(df['Close'] / df['Close'].shift(1))
    df['rsi_14']      = compute_rsi(df['Close'])
    df['macd'], df['macd_signal'] = compute_macd(df['Close'])
    df['bb_upper'], df['bb_lower'] = compute_bb(df['Close'])
    df['sma_20']      = df['Close'].rolling(20).mean()
    df['volume_log']  = np.log1p(df['Volume'])

    keep = ['Date','Close','log_return','rsi_14','macd','macd_signal','bb_upper','bb_lower','sma_20','volume_log']
    df = df[keep].dropna().reset_index(drop=True)
    df.to_csv(f'processed_data/{ticker}_features.csv', index=False)
    print(f'  {ticker}: {len(df)} rows')

print('✓ Feature engineering complete')

## Cell 7 — Run FinBERT Sentiment (uses GPU, ~2–3 hrs for all 5 tickers)

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import pipeline
from pathlib import Path
from tqdm.notebook import tqdm

TICKERS   = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'GOOGL']
BATCH_SIZE = 32

device = 0 if torch.cuda.is_available() else -1
print(f'Device: {"GPU" if device == 0 else "CPU (will be slow!)"}\n')

print('Loading FinBERT model (~500 MB download, cached after first run)...')
pipe = pipeline(
    'text-classification',
    model='ProsusAI/finbert',
    return_all_scores=True,
    device=device,
    batch_size=BATCH_SIZE,
)
print('✓ FinBERT loaded\n')

def score_batch(texts):
    cleaned = [str(t)[:512] if pd.notna(t) else '' for t in texts]
    results = pipe(cleaned)
    return [r['positive']['score'] - r['negative']['score']
            if isinstance(r, dict)
            else next(x['score'] for x in r if x['label']=='positive') -
                 next(x['score'] for x in r if x['label']=='negative')
            for r in [{item['label'].lower(): item['score'] for item in res} for res in results]]

for ticker in TICKERS:
    path = Path(f'raw_data/news/{ticker}_headlines.csv')
    if not path.exists():
        print(f'  [{ticker}] No headlines file — skipping. Sentiment will be 0.')
        # Write empty sentinel so build_sequences doesn't error
        pd.DataFrame(columns=['date','mean_score','headline_count','std_score']).to_csv(
            f'sentiment_scores/{ticker}_sentiment.csv', index=False)
        continue

    df = pd.read_csv(path, parse_dates=['date']).dropna(subset=['headline']).reset_index(drop=True)
    if len(df) == 0:
        print(f'  [{ticker}] Empty headlines file — skipping')
        pd.DataFrame(columns=['date','mean_score','headline_count','std_score']).to_csv(
            f'sentiment_scores/{ticker}_sentiment.csv', index=False)
        continue

    print(f'\n{ticker} — {len(df):,} headlines')
    headlines = df['headline'].tolist()
    all_scores = []

    for i in tqdm(range(0, len(headlines), BATCH_SIZE), desc=ticker):
        all_scores.extend(score_batch(headlines[i:i+BATCH_SIZE]))

    df['score'] = all_scores
    df['date'] = pd.to_datetime(df['date']).dt.normalize()

    daily = df.groupby('date')['score'].agg(
        mean_score='mean', headline_count='count', std_score='std'
    ).reset_index()
    daily['std_score'] = daily['std_score'].fillna(0.0)

    daily.to_csv(f'sentiment_scores/{ticker}_sentiment.csv', index=False)
    print(f'  ✓ {len(daily)} trading days saved | mean={daily["mean_score"].mean():.3f}')

print('\n✓ FinBERT complete')

## Cell 8 — Build Sliding-Window Sequences

In [ ]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

TICKERS  = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'GOOGL']
LOOKBACK = 60
FEATURE_COLS = [
    'Close','log_return','rsi_14','macd','macd_signal',
    'bb_upper','bb_lower','sma_20','volume_log','sentiment_score',
]

for ticker in TICKERS:
    price_df = pd.read_csv(f'processed_data/{ticker}_features.csv', parse_dates=['Date'])

    sent_path = Path(f'sentiment_scores/{ticker}_sentiment.csv')
    if sent_path.exists():
        sent_df = pd.read_csv(sent_path, parse_dates=['date']).rename(
            columns={'date':'Date','mean_score':'sentiment_score'})
        df = price_df.merge(sent_df[['Date','sentiment_score']], on='Date', how='left')
    else:
        df = price_df.copy()
        df['sentiment_score'] = 0.0

    df['sentiment_score'] = (
        df['sentiment_score']
        .fillna(df['sentiment_score'].rolling(30, min_periods=1).mean())
        .fillna(0.0)
    )
    df = df.sort_values('Date').reset_index(drop=True)

    features = df[FEATURE_COLS].values.astype(np.float32)
    dates    = df['Date'].values
    n        = len(features)
    train_end, val_end = int(n*0.70), int(n*0.85)

    # Fit feature scaler on training data only
    scaler = MinMaxScaler()
    scaler.fit(features[:train_end])
    features_scaled = scaler.transform(features)
    joblib.dump(scaler, f'processed_data/{ticker}_scaler.pkl')

    # Separate scaler for Close only — used to inverse-transform predictions back to dollars
    close_scaler = MinMaxScaler()
    close_scaler.fit(features[:train_end, 0:1])
    joblib.dump(close_scaler, f'processed_data/{ticker}_close_scaler.pkl')

    # Use SCALED close as target so model learns in 0-1 space
    X, y = [], []
    for i in range(LOOKBACK, n):
        X.append(features_scaled[i-LOOKBACK:i])
        y.append(features_scaled[i, 0])   # scaled Close, NOT raw dollars
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.float32)
    seq_dates = dates[LOOKBACK:]

    t_end = train_end - LOOKBACK
    v_end = val_end   - LOOKBACK

    np.savez(
        f'processed_data/{ticker}_sequences.npz',
        X_train=X[:t_end],      y_train=y[:t_end],
        X_val  =X[t_end:v_end], y_val  =y[t_end:v_end],
        X_test =X[v_end:],      y_test =y[v_end:],
        dates_train=seq_dates[:t_end],
        dates_val  =seq_dates[t_end:v_end],
        dates_test =seq_dates[v_end:],
    )
    print(f'  {ticker}: Train={t_end} | Val={v_end-t_end} | Test={len(X)-v_end}')

print('✓ Sequences built')

## Cell 9 — Train LSTM + Sentiment (Proposed Model)

In [ ]:
import numpy as np
import pandas as pd
import json
import joblib
import tensorflow as tf
from pathlib import Path

TICKERS    = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'GOOGL']
LOOKBACK   = 60
N_FEATURES = 10

def build_lstm(lookback, n_features):
    m = tf.keras.Sequential([
        tf.keras.layers.LSTM(128, return_sequences=True,
                             input_shape=(lookback, n_features),
                             kernel_regularizer=tf.keras.regularizers.L2(1e-4)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.LSTM(64, kernel_regularizer=tf.keras.regularizers.L2(1e-4)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='mse', metrics=['mae'])
    return m

Path('saved_models').mkdir(exist_ok=True)
Path('results/training_history').mkdir(parents=True, exist_ok=True)
Path('results/predictions').mkdir(parents=True, exist_ok=True)

for ticker in TICKERS:
    print(f'\n{"="*50}\nLSTM-Sentiment | {ticker}')
    d = np.load(f'processed_data/{ticker}_sequences.npz', allow_pickle=True)
    close_scaler = joblib.load(f'processed_data/{ticker}_close_scaler.pkl')

    model = build_lstm(LOOKBACK, N_FEATURES)
    history = model.fit(
        d['X_train'], d['y_train'],
        validation_data=(d['X_val'], d['y_val']),
        epochs=150, batch_size=32,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-5),
        ],
        verbose=1,
    )
    model.save(f'saved_models/{ticker}_lstm_sentiment.keras')
    with open(f'results/training_history/{ticker}_lstm_sentiment.json', 'w') as f:
        json.dump(history.history, f)

    # Inverse-transform both actual and predicted back to dollar prices
    y_pred_scaled = model.predict(d['X_test']).flatten()
    y_pred = close_scaler.inverse_transform(y_pred_scaled.reshape(-1,1)).flatten()
    y_test = close_scaler.inverse_transform(d['y_test'].reshape(-1,1)).flatten()

    pd.DataFrame({
        'date': d['dates_test'],
        'actual': y_test,
        'lstm_sentiment_pred': y_pred,
    }).to_csv(f'results/predictions/{ticker}_test_predictions.csv', index=False)
    print(f'  ✓ Saved | price range: ${y_pred.min():.0f}–${y_pred.max():.0f}')

print('\n✓ LSTM-Sentiment training complete')

## Cell 10 — Train LSTM-Only Baseline

In [ ]:
import numpy as np
import pandas as pd
import json
import joblib
import tensorflow as tf

TICKERS        = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'GOOGL']
N_FEAT_NO_SENT = 9

def build_lstm_only():
    m = tf.keras.Sequential([
        tf.keras.layers.LSTM(128, return_sequences=True, input_shape=(60, N_FEAT_NO_SENT),
                             kernel_regularizer=tf.keras.regularizers.L2(1e-4)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.LSTM(64, kernel_regularizer=tf.keras.regularizers.L2(1e-4)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='mse', metrics=['mae'])
    return m

for ticker in TICKERS:
    print(f'\nLSTM-Only | {ticker}')
    d = np.load(f'processed_data/{ticker}_sequences.npz', allow_pickle=True)
    close_scaler = joblib.load(f'processed_data/{ticker}_close_scaler.pkl')

    X_train = d['X_train'][:,:,:N_FEAT_NO_SENT]
    X_val   = d['X_val'][:,:,:N_FEAT_NO_SENT]
    X_test  = d['X_test'][:,:,:N_FEAT_NO_SENT]

    model = build_lstm_only()
    history = model.fit(
        X_train, d['y_train'],
        validation_data=(X_val, d['y_val']),
        epochs=150, batch_size=32,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-5),
        ],
        verbose=1,
    )
    model.save(f'saved_models/{ticker}_lstm_only.keras')
    with open(f'results/training_history/{ticker}_lstm_only.json', 'w') as f:
        json.dump(history.history, f)

    # Inverse-transform back to dollar prices
    y_pred_scaled = model.predict(X_test).flatten()
    y_pred = close_scaler.inverse_transform(y_pred_scaled.reshape(-1,1)).flatten()

    pred_path = f'results/predictions/{ticker}_test_predictions.csv'
    df = pd.read_csv(pred_path, parse_dates=['date'])
    df['lstm_only_pred'] = y_pred
    df.to_csv(pred_path, index=False)
    print(f'  ✓ Saved | price range: ${y_pred.min():.0f}–${y_pred.max():.0f}')

print('\n✓ LSTM-Only training complete')

## Cell 11 — Train ARIMA Baseline

In [ ]:
import numpy as np
import pandas as pd
import joblib
from pmdarima import auto_arima

TICKERS = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'GOOGL']

for ticker in TICKERS:
    print(f'\nARIMA | {ticker}')
    # Use the same feature CSV that sequences were built from — same row count, same split indices
    df = pd.read_csv(f'processed_data/{ticker}_features.csv', parse_dates=['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    prices = df['Close'].values.astype(float)
    n = len(prices)
    train_end, val_end = int(n*0.70), int(n*0.85)

    # Read the predictions CSV first so we know exactly how many test rows to predict
    pred_path = f'results/predictions/{ticker}_test_predictions.csv'
    df_p = pd.read_csv(pred_path, parse_dates=['date'])
    n_test = len(df_p)

    model = auto_arima(prices[:train_end], max_p=5, max_q=5, max_d=2,
                       seasonal=False, stepwise=True, error_action='ignore',
                       suppress_warnings=True, information_criterion='aic')
    print(f'  Order: {model.order}')
    model.update(prices[train_end:val_end])
    y_pred = model.predict(n_periods=n_test)

    joblib.dump(model, f'saved_models/{ticker}_arima.pkl')

    df_p['arima_pred'] = y_pred
    df_p.to_csv(pred_path, index=False)
    print(f'  ✓ Saved ({n_test} predictions)')

print('\n✓ ARIMA training complete')

## Cell 12 — Train SVR Baseline

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

TICKERS = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'GOOGL']
N_FEAT_NO_SENT = 9

for ticker in TICKERS:
    print(f'\nSVR | {ticker}')
    d = np.load(f'processed_data/{ticker}_sequences.npz', allow_pickle=True)
    X_train = d['X_train'][:,:,:N_FEAT_NO_SENT].reshape(len(d['X_train']), -1)
    X_val   = d['X_val'][:,:,:N_FEAT_NO_SENT].reshape(len(d['X_val']), -1)
    X_test  = d['X_test'][:,:,:N_FEAT_NO_SENT].reshape(len(d['X_test']), -1)

    scaler = StandardScaler()
    X_fit  = scaler.fit_transform(np.vstack([X_train, X_val]))
    X_test_s = scaler.transform(X_test)
    y_fit  = np.concatenate([d['y_train'], d['y_val']])

    print(f'  Fitting SVR on {len(X_fit)} samples...')
    model = SVR(kernel='rbf', C=100, gamma=0.001, epsilon=0.1)
    model.fit(X_fit, y_fit)
    y_pred = model.predict(X_test_s)

    joblib.dump({'svr': model, 'scaler': scaler}, f'saved_models/{ticker}_svr.pkl')

    pred_path = f'results/predictions/{ticker}_test_predictions.csv'
    df_p = pd.read_csv(pred_path, parse_dates=['date'])
    df_p['svr_pred'] = y_pred
    df_p.to_csv(pred_path, index=False)
    print(f'  ✓ Saved')

print('\n✓ SVR training complete')

## Cell 13 — Evaluate All Models → metrics.json

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

TICKERS   = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'GOOGL']
MODEL_COLS = {
    'LSTM-Sentiment': 'lstm_sentiment_pred',
    'LSTM-Only':      'lstm_only_pred',
    'ARIMA':          'arima_pred',
    'SVR':            'svr_pred',
}

def metrics(y_true, y_pred, ticker, model):
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    mape = float(np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100)
    da   = float(np.mean((np.diff(y_true) > 0) == (np.diff(y_pred) > 0)) * 100)
    return {'ticker': ticker, 'model': model,
            'rmse': round(rmse,4), 'mae': round(mae,4),
            'mape': round(mape,4), 'directional_accuracy': round(da,2)}

all_metrics = []
for ticker in TICKERS:
    df = pd.read_csv(f'results/predictions/{ticker}_test_predictions.csv', parse_dates=['date'])
    y_true = df['actual'].values
    for model_name, col in MODEL_COLS.items():
        if col not in df.columns: continue
        y_pred = df[col].values
        mask = ~np.isnan(y_pred)
        m = metrics(y_true[mask], y_pred[mask], ticker, model_name)
        all_metrics.append(m)
        print(f'  {ticker:5s} | {model_name:18s} | RMSE={m["rmse"]:7.2f} | MAE={m["mae"]:7.2f} | MAPE={m["mape"]:5.2f}% | DirAcc={m["directional_accuracy"]:5.1f}%')

with open('results/metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)

print(f'\n✓ metrics.json saved ({len(all_metrics)} records)')

## Cell 14 — Package & Download Everything

## Cell 14 — Launch Dashboard (runs entirely in Colab, no local setup needed)

In [ ]:
!pip install gradio plotly -q

import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import gradio as gr
import joblib
from pathlib import Path

TICKERS    = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'GOOGL']
MODEL_COLS = {
    'LSTM+Sentiment (Proposed)': 'lstm_sentiment_pred',
    'LSTM-Only':                 'lstm_only_pred',
    'ARIMA':                     'arima_pred',
    'SVR':                       'svr_pred',
}
COLOURS = {
    'LSTM+Sentiment (Proposed)': '#10b981',
    'LSTM-Only':                 '#ef4444',
    'ARIMA':                     '#f59e0b',
    'SVR':                       '#8b5cf6',
}

# ── Load all data once ────────────────────────────────────────────────────────
predictions, sentiments, metrics_list = {}, {}, []

for t in TICKERS:
    p = Path(f'results/predictions/{t}_test_predictions.csv')
    if p.exists():
        predictions[t] = pd.read_csv(p, parse_dates=['date'])
    s = Path(f'sentiment_scores/{t}_sentiment.csv')
    if s.exists():
        sentiments[t] = pd.read_csv(s, parse_dates=['date'])

m_path = Path('results/metrics.json')
if m_path.exists():
    with open(m_path) as f:
        metrics_list = json.load(f)

# ── Chart builders ────────────────────────────────────────────────────────────
def make_price_chart(ticker):
    df = predictions.get(ticker)
    if df is None:
        return go.Figure().update_layout(title='No prediction data yet')

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df['date'], y=df['actual'],
                             name='Actual', line=dict(color='#1e40af', width=2)))
    for label, col in MODEL_COLS.items():
        if col in df.columns:
            fig.add_trace(go.Scatter(
                x=df['date'], y=df[col], name=label,
                line=dict(color=COLOURS[label], width=1.5, dash='dash'),
            ))
    fig.update_layout(
        title=f'{ticker} — Actual vs Predicted (Test Set)',
        xaxis_title='Date', yaxis_title='Price (USD)',
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
        hovermode='x unified', height=420,
        plot_bgcolor='#f8fafc', paper_bgcolor='white',
        yaxis=dict(tickprefix='$'),
    )
    return fig


def make_sentiment_chart(ticker):
    df = sentiments.get(ticker)
    if df is None:
        return go.Figure().update_layout(title='No sentiment data yet')

    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_trace(go.Bar(
        x=df['date'], y=df['headline_count'],
        name='Headline Count', marker_color='#e2e8f0', opacity=0.6,
    ), secondary_y=True)
    fig.add_trace(go.Scatter(
        x=df['date'], y=df['mean_score'],
        name='Sentiment Score', line=dict(color='#10b981', width=1.5),
        fill='tozeroy', fillcolor='rgba(16,185,129,0.1)',
    ), secondary_y=False)
    fig.add_hline(y=0, line_dash='dash', line_color='#94a3b8')
    fig.update_layout(
        title=f'{ticker} — FinBERT Daily Sentiment Score',
        hovermode='x unified', height=320,
        plot_bgcolor='#f8fafc', paper_bgcolor='white',
        yaxis=dict(title='Sentiment Score', range=[-1, 1]),
        yaxis2=dict(title='Headline Count'),
    )
    return fig


def make_metrics_table(ticker):
    rows = []
    for label in MODEL_COLS:
        entry = next((m for m in metrics_list if m['ticker'] == ticker and m['model'] == label), None)
        if entry:
            rows.append([
                label,
                f"{entry['rmse']:.2f}",
                f"{entry['mae']:.2f}",
                f"{entry['mape']:.2f}%",
                f"{entry['directional_accuracy']:.1f}%",
            ])
        else:
            rows.append([label, '—', '—', '—', '—'])

    df = pd.DataFrame(rows, columns=['Model', 'RMSE', 'MAE', 'MAPE', 'Dir Acc'])
    return df


def make_rmse_bar():
    if not metrics_list:
        return go.Figure().update_layout(title='No metrics yet')
    df = pd.DataFrame(metrics_list)
    fig = px.bar(df, x='ticker', y='rmse', color='model',
                 barmode='group', title='RMSE by Ticker & Model',
                 color_discrete_map={k: v for k, v in zip(
                     ['LSTM+Sentiment (Proposed)', 'LSTM-Only', 'ARIMA', 'SVR'],
                     ['#10b981', '#ef4444', '#f59e0b', '#8b5cf6'])},
                 labels={'rmse': 'RMSE ($)', 'ticker': 'Ticker', 'model': 'Model'})
    fig.update_layout(height=380, plot_bgcolor='#f8fafc', paper_bgcolor='white')
    return fig


def predict_tomorrow(ticker):
    try:
        import yfinance as yf
        import tensorflow as tf

        model_path = Path(f'saved_models/{ticker}_lstm_sentiment.keras')
        scaler_path = Path(f'processed_data/{ticker}_scaler.pkl')
        if not model_path.exists():
            return f'Model for {ticker} not found. Make sure training completed.'
        if not scaler_path.exists():
            return f'Scaler for {ticker} not found.'

        model  = tf.keras.models.load_model(str(model_path))
        scaler = joblib.load(str(scaler_path))

        df = yf.download(ticker, period='6mo', auto_adjust=True, progress=False)
        df.reset_index(inplace=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [c[0] for c in df.columns]
        df['Close']  = pd.to_numeric(df['Close'],  errors='coerce').ffill()
        df['Volume'] = pd.to_numeric(df['Volume'], errors='coerce').fillna(0)
        df = df.sort_values('Date').reset_index(drop=True)

        def rsi(s, p=14):
            d=s.diff(); g=d.clip(lower=0); l=-d.clip(upper=0)
            return 100-(100/(1+g.ewm(com=p-1,min_periods=p).mean()/(l.ewm(com=p-1,min_periods=p).mean()+1e-10)))
        def macd(s):
            m=s.ewm(span=12,adjust=False).mean()-s.ewm(span=26,adjust=False).mean()
            return m, m.ewm(span=9,adjust=False).mean()

        df['log_return'] = np.log(df['Close']/df['Close'].shift(1))
        df['rsi_14']     = rsi(df['Close'])
        df['macd'], df['macd_signal'] = macd(df['Close'])
        sma = df['Close'].rolling(20).mean(); std = df['Close'].rolling(20).std()
        df['bb_upper'], df['bb_lower'] = sma+2*std, sma-2*std
        df['sma_20']     = sma
        df['volume_log'] = np.log1p(df['Volume'])

        sent = sentiments.get(ticker)
        sent_val = sent['mean_score'].tail(30).mean() if sent is not None else 0.0
        df['sentiment_score'] = sent_val
        df = df.dropna().reset_index(drop=True)

        if len(df) < 60:
            return f'Not enough recent data ({len(df)} rows, need 60).'

        FEAT = ['Close','log_return','rsi_14','macd','macd_signal','bb_upper','bb_lower','sma_20','volume_log','sentiment_score']
        features = df[FEAT].values[-60:].astype(np.float32)
        features_scaled = scaler.transform(features)
        X = features_scaled.reshape(1, 60, 10)

        pred  = float(model.predict(X, verbose=0)[0][0])
        last  = float(df['Close'].iloc[-1])
        date  = str(df['Date'].iloc[-1])[:10]
        diff  = pred - last
        pct   = diff / last * 100
        arrow = '▲' if diff > 0 else '▼'

        return (
            f"Last close ({date}):    ${last:.2f}\n"
            f"Predicted tomorrow:  ${pred:.2f}\n"
            f"Direction:           {arrow} {'+' if diff>0 else ''}{pct:.2f}%\n\n"
            f"(For research purposes only — not financial advice)"
        )
    except Exception as e:
        return f'Error: {e}'


# ── Gradio UI ─────────────────────────────────────────────────────────────────
with gr.Blocks(title='StockSentinel', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# StockSentinel\n### LSTM + FinBERT Sentiment Analysis | FYP — Aniq Amrin bin Azri, UiTM')

    with gr.Row():
        ticker_dd = gr.Dropdown(choices=TICKERS, value='AAPL', label='Select Ticker')

    with gr.Tabs():
        with gr.TabItem('Price Predictions'):
            price_plot = gr.Plot()
            metrics_df = gr.Dataframe(label='Model Comparison (selected ticker)',
                                      headers=['Model','RMSE','MAE','MAPE','Dir Acc'])

        with gr.TabItem('Sentiment Analysis'):
            sent_plot = gr.Plot()

        with gr.TabItem('Overall Comparison'):
            rmse_plot = gr.Plot(value=make_rmse_bar())

        with gr.TabItem('Predict Tomorrow'):
            gr.Markdown('Fetches latest data from Yahoo Finance and runs the saved LSTM+Sentiment model.')
            predict_btn = gr.Button('Run Prediction', variant='primary')
            predict_out = gr.Textbox(label='Result', lines=6)
            predict_btn.click(predict_tomorrow, inputs=ticker_dd, outputs=predict_out)

    def update(ticker):
        return make_price_chart(ticker), make_metrics_table(ticker), make_sentiment_chart(ticker)

    ticker_dd.change(update, inputs=ticker_dd, outputs=[price_plot, metrics_df, sent_plot])
    demo.load(update, inputs=ticker_dd, outputs=[price_plot, metrics_df, sent_plot])

demo.launch(share=True)  # share=True gives you a public link valid for 72 hours

In [ ]:
import shutil
from google.colab import files
from pathlib import Path

# Bundle everything the local project needs
print('Zipping results...')
shutil.make_archive('stocksentinel_results', 'zip', '.', 'results')
print('Zipping sentiment scores...')
shutil.make_archive('sentiment_scores', 'zip', '.', 'sentiment_scores')
print('Zipping saved models...')
shutil.make_archive('saved_models', 'zip', '.', 'saved_models')
print('Zipping processed data (sequences + scalers)...')
shutil.make_archive('processed_data', 'zip', '.', 'processed_data')

print('\nDownloading zips to your computer...')
for fname in ['stocksentinel_results.zip', 'sentiment_scores.zip', 'saved_models.zip', 'processed_data.zip']:
    print(f'  Downloading {fname}...')
    files.download(fname)

print('\n✓ Done! Unzip each file into your project root on your local machine.')
print('   Then start the backend + frontend and the dashboard will have all data.')

## ✓ After downloading

Unzip all 4 files into `C:\Users\ANIQ AMRIN AZRI\Desktop\LTSM_STOCK\`:
```
stocksentinel_results.zip  →  results/
sentiment_scores.zip       →  sentiment_scores/
saved_models.zip           →  saved_models/
processed_data.zip         →  processed_data/
```

Then on your local machine:
```bash
cd backend && uvicorn main:app --reload
cd frontend && npm run dev
```
Open http://localhost:5173 — the dashboard will show all charts and predictions.